In [1]:
import kagglehub

original_path = kagglehub.dataset_download("xdxd003/ff-c23", path="FaceForensics++_C23/original")

deepfakes_path = kagglehub.dataset_download("xdxd003/ff-c23", path="FaceForensics++_C23/Deepfakes")

print(f"Original files at: {original_path}")
print(f"Deepfakes files at: {deepfakes_path}")

Original files at: /kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23/original
Deepfakes files at: /kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23/Deepfakes


In [2]:
import os
original_count = len(os.listdir('/kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23/original'))
deepfake_count = len(os.listdir('/kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23/Deepfakes'))

print(f"Original videos: {original_count}")
print(f"Deepfake videos: {deepfake_count}")

Original videos: 1000
Deepfake videos: 1000


In [3]:
import os

base = '/kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23/'

for folder in ['original', 'Deepfakes']:
    path = os.path.join(base, folder)
    files = os.listdir(path)
    print(f"\n{folder}/")
    print(f"  Total files: {len(files)}")
    print(f"  First 5: {files[:5]}")
    
    # Check one file to understand format
    sample = os.path.join(path, files[0])
    size_mb = os.path.getsize(sample) / (1024*1024)
    print(f"  Sample file: {files[0]}")
    print(f"  Sample size: {size_mb:.1f} MB")
    print(f"  Extension: {os.path.splitext(files[0])[1]}")


original/
  Total files: 1000
  First 5: ['123.mp4', '738.mp4', '479.mp4', '660.mp4', '565.mp4']
  Sample file: 123.mp4
  Sample size: 1.2 MB
  Extension: .mp4

Deepfakes/
  Total files: 1000
  First 5: ['479_706.mp4', '481_469.mp4', '184_205.mp4', '315_322.mp4', '645_688.mp4']
  Sample file: 479_706.mp4
  Sample size: 4.6 MB
  Extension: .mp4


In [4]:
# Check GPU
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# Check FFmpeg
import subprocess
result = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True)
print("\nFFmpeg:", result.stdout.split('\n')[0])

# Check if RetinaFace is available
try:
    from retinaface import RetinaFace
    print("\nRetinaFace: available")
except:
    print("\nRetinaFace: NOT installed")

# Check torchvision for ResNeXt50
try:
    from torchvision.models import resnext50_32x4d
    print("ResNeXt50: available")
except:
    print("ResNeXt50: NOT available")

# Check OpenCV
import cv2
print("OpenCV:", cv2.__version__)

CUDA available: True
Device: Tesla T4

FFmpeg: ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers

RetinaFace: NOT installed
ResNeXt50: available
OpenCV: 4.13.0


In [5]:
import subprocess
result = subprocess.run(
    ['pip', 'install', 'retina-face', '-q'],
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)

# Verify
from retinaface import RetinaFace
print("RetinaFace installed successfully!")

# Do a quick sanity check that it can load its model
import numpy as np
dummy = np.zeros((112, 112, 3), dtype=np.uint8)
result = RetinaFace.detect_faces(dummy)
print("RetinaFace inference check passed!")

2026-04-18 20:05:05.043047: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776542705.283953      22 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776542705.354649      22 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776542705.911971      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776542705.912013      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776542705.912016      22 computation_placer.cc:177] computation placer alr

RetinaFace installed successfully!


I0000 00:00:1776542726.057857      22 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1776542726.060850      22 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/retinaface.h5
To: /root/.deepface/weights/retinaface.h5


26-04-18 20:05:28 - Directory /root/.deepface created
26-04-18 20:05:28 - Directory /root/.deepface/weights created
26-04-18 20:05:28 - retinaface.h5 will be downloaded from the url https://github.com/serengil/deepface_models/releases/download/v1.0/retinaface.h5


100%|██████████| 119M/119M [00:00<00:00, 165MB/s] 
I0000 00:00:1776542731.411058      80 cuda_dnn.cc:529] Loaded cuDNN version 91002


RetinaFace inference check passed!


In [6]:
import os
import cv2
import numpy as np
import torch
import torchvision.transforms as transforms
from torchvision.models import resnext50_32x4d
from retinaface import RetinaFace
import subprocess
import tempfile
import shutil
from tqdm import tqdm

torch.cuda.empty_cache()

# Paths
BASE_DIR = '/kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23/'
OUTPUT_DIR = '/kaggle/working/features'
ORIGINAL_DIR = os.path.join(BASE_DIR, 'original')
DEEPFAKE_DIR = os.path.join(BASE_DIR, 'Deepfakes')

# Output folders
os.makedirs(os.path.join(OUTPUT_DIR, 'original'), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'deepfakes'), exist_ok=True)

# Config
FPS = 10
FACE_SIZE = (224, 224)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

Using device: cuda


In [7]:
# Load pretrained ResNeXt-50 and remove classification head
resnext = resnext50_32x4d(pretrained=True)
# Remove the final FC layer so we get 2048-dim embeddings
feature_extractor = torch.nn.Sequential(*list(resnext.children())[:-1])
feature_extractor = feature_extractor.to(DEVICE)
feature_extractor.eval()

# ImageNet normalization
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize(FACE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("ResNeXt-50 loaded and ready")

Downloading: "https://download.pytorch.org/models/resnext50_32x4d-7cdf4587.pth" to /root/.cache/torch/hub/checkpoints/resnext50_32x4d-7cdf4587.pth


100%|██████████| 95.8M/95.8M [00:01<00:00, 55.2MB/s]


ResNeXt-50 loaded and ready


In [8]:
def extract_frames_ffmpeg(video_path, fps=10):
    temp_dir = tempfile.mkdtemp()
    output_pattern = os.path.join(temp_dir, 'frame_%04d.png')
    
    cmd = [
        'ffmpeg', '-i', video_path,
        '-vf', f'fps={fps},hqdn3d=1.5:1.5:6:6',  # fps + light denoise
        '-pix_fmt', 'rgb24',
        '-q:v', '2',
        output_pattern,
        '-loglevel', 'error'
    ]
    subprocess.run(cmd, check=True)
    
    frames = []
    frame_files = sorted(os.listdir(temp_dir))
    for fname in frame_files:
        if fname.endswith('.png'):
            fpath = os.path.join(temp_dir, fname)
            frame = cv2.imread(fpath)
            if frame is not None:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frames.append(frame)
    
    shutil.rmtree(temp_dir)
    return frames


def detect_and_align_face(frame):
    try:
        faces = RetinaFace.detect_faces(frame)
        if not isinstance(faces, dict) or len(faces) == 0:
            return None
        
        # Take the face with highest confidence
        best_face = max(faces.values(), key=lambda x: x['score'])
        
        # Get landmarks (left eye, right eye, nose, left mouth, right mouth)
        landmarks = best_face['landmarks']
        left_eye = np.array(landmarks['left_eye'])
        right_eye = np.array(landmarks['right_eye'])
        
        # Compute angle for alignment
        dy = right_eye[1] - left_eye[1]
        dx = right_eye[0] - left_eye[0]
        angle = np.degrees(np.arctan2(dy, dx))
        
        # Bounding box crop
        bbox = best_face['facial_area']
        x1, y1, x2, y2 = bbox
        
        # Add margin around face
        margin = int(0.2 * max(x2 - x1, y2 - y1))
        h, w = frame.shape[:2]
        x1 = max(0, x1 - margin)
        y1 = max(0, y1 - margin)
        x2 = min(w, x2 + margin)
        y2 = min(h, y2 + margin)
        
        face_crop = frame[y1:y2, x1:x2]
        if face_crop.size == 0:
            return None
        
        # Rotate for alignment
        center = (face_crop.shape[1] // 2, face_crop.shape[0] // 2)
        rot_matrix = cv2.getRotationMatrix2D(center, angle, 1.0)
        aligned = cv2.warpAffine(face_crop, rot_matrix, 
                                  (face_crop.shape[1], face_crop.shape[0]))
        
        # Resize to target
        aligned = cv2.resize(aligned, FACE_SIZE)
        return aligned
    
    except Exception:
        return None


def extract_embedding(face_img):
    tensor = transform(face_img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        embedding = feature_extractor(tensor)
    return embedding.squeeze().cpu().numpy()  # shape: (2048,)


def process_video(video_path):
    frames = extract_frames_ffmpeg(video_path, fps=FPS)
    if len(frames) == 0:
        return None
    
    embeddings = []
    for frame in frames:
        face = detect_and_align_face(frame)
        if face is None:
            continue
        embedding = extract_embedding(face)
        embeddings.append(embedding)
    
    if len(embeddings) == 0:
        return None
    
    return np.array(embeddings)  # shape: (num_frames, 2048)

In [9]:
def process_folder_range(input_folder, output_folder, label, start_idx, end_idx):
    video_files = sorted([f for f in os.listdir(input_folder) if f.endswith('.mp4')])
    video_files = video_files[start_idx:end_idx]  # slice for this session
    
    skipped = 0
    failed = 0
    processed = 0
    
    print(f"Processing {label} videos {start_idx} to {end_idx} ({len(video_files)} videos)")
    
    for fname in tqdm(video_files, desc=f"Processing {label}"):
        video_name = os.path.splitext(fname)[0]
        output_path = os.path.join(output_folder, f"{video_name}.npy")
        
        # This makes the process resumable: skip if already done
        if os.path.exists(output_path):
            skipped += 1
            continue
        
        video_path = os.path.join(input_folder, fname)
        
        try:
            embeddings = process_video(video_path)
            if embeddings is not None:
                np.save(output_path, embeddings)
                processed += 1
            else:
                print(f"  [WARN] No faces found: {fname}")
                failed += 1
        except Exception as e:
            print(f"  [ERROR] {fname}: {e}")
            failed += 1
    
    print(f"\n{label} [{start_idx}:{end_idx}] done — processed: {processed}, skipped: {skipped}, failed: {failed}")



# We will change this for each session to be safe from session limit on Kaggle 
SESSION = 5  # Change to 1, 2, 3, 4 and 5

# Each session does 200 original + 200 deepfakes = 400 total
RANGES = {
    1: (0, 200),
    2: (200, 400),
    3: (400, 600),
    4: (600, 800),
    5: (800, 1000),
}
start_idx, end_idx = RANGES[SESSION]

process_folder_range(
    ORIGINAL_DIR,
    os.path.join(OUTPUT_DIR, 'original'),
    label='original',
    start_idx=start_idx,
    end_idx=end_idx
)

process_folder_range(
    DEEPFAKE_DIR,
    os.path.join(OUTPUT_DIR, 'deepfakes'),
    label='deepfakes',
    start_idx=start_idx,
    end_idx=end_idx
)

Processing original videos 800 to 1000 (200 videos)


Processing original: 100%|██████████| 200/200 [2:22:20<00:00, 42.70s/it]



original [800:1000] done — processed: 200, skipped: 0, failed: 0
Processing deepfakes videos 800 to 1000 (200 videos)


Processing deepfakes: 100%|██████████| 200/200 [2:22:34<00:00, 42.77s/it]


deepfakes [800:1000] done — processed: 200, skipped: 0, failed: 0


In [10]:
# Verify output
orig_features = os.listdir(os.path.join(OUTPUT_DIR, 'original'))
fake_features = os.listdir(os.path.join(OUTPUT_DIR, 'deepfakes'))
print(f"Original feature files: {len(orig_features)}")
print(f"Deepfake feature files: {len(fake_features)}")


# Zip everything
shutil.make_archive('/kaggle/working/features', 'zip', OUTPUT_DIR)
print("Zipped! - /kaggle/working/features.zip")

Original feature files: 200
Deepfake feature files: 200
Zipped! - /kaggle/working/features.zip
